# Option 2 finish-up run

Fully self-contained. Runtime -> Run all. No editing needed.

All GPU/generation steps (extract, behavior, causal, steering, judge) are
resumable - already-completed work is skipped, not reprocessed - so this is
safe to run end-to-end even if some stages are already done on this Drive
account. The judge (step 7) is the one exception: it always rescans and
rescoring everything fresh, ~30-45 min, every time it runs.

Everything from step 8 on is CPU-only, and step 11 builds the actual human
annotation packet automatically - no separate manual follow-up needed.

## 0. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Clone/pin the repo, install deps, bind results/ + HF cache to Drive

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/urosavurdic/dpo-safety-representations.git'
REPO_DIR = '/content/dpo-safety-representations'
BRANCH = 'agent/c-quadrant-end-to-end-e0e2317a'
PINNED_COMMIT = '101cfd0d7f70993476c2342a8729692d033bd72e'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
subprocess.run(['git', 'fetch', 'origin'], check=True)
subprocess.run(['git', 'checkout', PINNED_COMMIT], check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
assert commit == PINNED_COMMIT, f'wrong commit: {commit} != {PINNED_COMMIT}'
print('checked out', commit)


In [ ]:
!pip -q install -r requirements.txt
# The judges load in 4-bit; without bitsandbytes both fail to load and the
# judge pass silently produces an all-"unavailable" output file.
!pip -q install -U "bitsandbytes>=0.46.1"
!pip uninstall -y torchao || true
!nvidia-smi


In [ ]:
import os, glob

# Pick the dpo_v2 folder that actually contains data. A stale empty
# MyDrive/dpo_v2 stub (left behind by an earlier failed bind()) otherwise
# shadows the real shared folder and you get "0/9" every fresh session.
candidates = ['/content/drive/MyDrive/dpo_v2']
candidates += sorted(glob.glob('/content/drive/.shortcut-targets-by-id/*/dpo_v2'))
candidates += sorted(glob.glob('/content/drive/Shareddrives/*/dpo_v2'))

real_root = None
for c in candidates:
    if glob.glob(os.path.join(c, 'results', 'activations', '*_final.npy')):
        real_root = c
        break

if real_root is None:
    raise SystemExit(
        'No dpo_v2 folder with real activations found. Checked:\n  '
        + '\n  '.join(candidates)
        + '\n\nFix: in Drive, open the shared dpo_v2 folder -> "Add shortcut to '
          'Drive", and delete any empty MyDrive/dpo_v2 stub shadowing it.'
    )

os.environ['DPO_DRIVE_ROOT'] = real_root
print('DPO_DRIVE_ROOT =', real_root)

from src.colab_persist import bind, status_line
info = bind()
print(status_line(info))
!python -m src.analysis.v2_pipeline status


## 1b. HuggingFace auth (REQUIRED for the judge, step 7)
StrongREJECT's base (`google/gemma-2b`) and `allenai/wildguard` are **gated** repos. Set a Colab secret named `HF_TOKEN`: click the key icon in the left sidebar -> Add new secret -> Name `HF_TOKEN`, Value = a Read token from an HF account that has **accepted the licences for both `google/gemma-2b` and `allenai/wildguard`** -> turn "Notebook access" ON. Do not paste the token into a cell.

In [ ]:
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))
    print('HF login OK')
except Exception as e:
    print('HF NOT authenticated:', repr(e))
    print('Steps 2-6 still run without it. Step 7 (the judge) WILL FAIL until '
          'the HF_TOKEN Colab secret is set - see the markdown above this cell.')


## 2. Extract activations for all 9 stages against the current 654-row benchmark
Already-bound stages are skipped automatically (checked per-stage against the live benchmark) - safe and cheap to run for all 9 every time, regardless of which stages this particular Drive account/mount can already see.

In [ ]:
!python -m src.analysis.v2_pipeline extract --stages M0 M1 M2 M3 M3_direct M1_alt M2_alt M3_alt M3_direct_alt
!python -m src.analysis.verify_activations


## 3. Refresh behavioral responses for all 9 stages
All 9, not just the originally-stale ones - M2/M3's v2 behavioral responses (needed for CF1) turned out to be missing from this Drive folder too, so they're folded in here instead of a separate follow-up pass. Already-complete stages are skipped.

In [ ]:
!python -m src.analysis.v2_pipeline behavior --stages M0 M1 M2 M3 M3_direct M1_alt M2_alt M3_alt M3_direct_alt


## 4. Refresh directions + probes for all 9 stages
`--force` matters: without it, `direction` silently no-ops because stale direction files already exist on disk.

In [ ]:
!python -m src.analysis.v2_pipeline direction --stages M0 M1 M2 M3 M3_direct M1_alt M2_alt M3_alt M3_direct_alt --force
!python -m src.analysis.v2_pipeline probes    --stages M0 M1 M2 M3 M3_direct M1_alt M2_alt M3_alt M3_direct_alt


## 5. Causal ablation for all 4 DPO-endpoint branches
Includes M3 too, not just the 3 missing branches - safe/cheap to re-run (skips if the output file already exists), and insurance against this Drive account not actually being able to see M3's already-completed output.

In [ ]:
!python -m src.analysis.v2_pipeline causal --stage M3 --conditions baseline ablated_AD ablated_random
!python -m src.analysis.v2_pipeline causal --stage M3_direct --conditions baseline ablated_AD ablated_random
!python -m src.analysis.v2_pipeline causal --stage M3_alt --conditions baseline ablated_AD ablated_random
!python -m src.analysis.v2_pipeline causal --stage M3_direct_alt --conditions baseline ablated_AD ablated_random


## 6. Steering for all 4 DPO-endpoint branches
Includes M3 and M3_alt too - `eval_steering_v2.py` never overwrites an existing coefficient file, so this is safe/cheap to re-run and the same insurance as step 5.

In [ ]:
!python -m src.analysis.v2_pipeline steering --stage M3 --alpha-coefficients 0.5 1.0 2.0
!python -m src.analysis.v2_pipeline steering --stage M3_alt --alpha-coefficients 0.5 1.0 2.0
!python -m src.analysis.v2_pipeline steering --stage M3_direct --alpha-coefficients 0.5 1.0 2.0
!python -m src.analysis.v2_pipeline steering --stage M3_direct_alt --alpha-coefficients 0.5 1.0 2.0


## 7. Judge: rebuild the consolidated manifest and re-score everything in scope (LAST GPU STEP)
`--from-results-dir results` rescans results/ and rebuilds the manifest itself, so it automatically picks up the 3 new causal branches' response files.

In [ ]:
!python -m src.analysis.behavioral_judges \
  --response-manifest results/manifests/consolidated_judge.json \
  --from-results-dir results \
  --out-dir results/behavioral_judges_v2 \
  --run-live --scope confirmatory


---
# Everything below is CPU-only

## 8. Confirmatory endpoints (CF1 + per-branch CF2) - auto-finds the newest judge output

In [ ]:
import glob
judged_files = sorted(glob.glob('results/behavioral_judges_v2/behavioral_judges_v2_*.json'))
assert judged_files, 'No judge output found - did the judge cell finish?'
latest_judged = judged_files[-1]
print('Using judge file:', latest_judged)
!python -m src.analysis.confirmatory_behavioral_endpoints \
  --judged {latest_judged} \
  --benchmark data/frozen_v2/benchmark_v2_20260826T212909Z.jsonl \
  --out results/summaries/confirmatory_endpoints.json


## 9. Full descriptive / geometric / decodability story (all 9 stages, all branches)

In [ ]:
!python -m src.analysis.subspace_geometry
!python -m src.analysis.projection_trajectory
!python -m src.analysis.direction_decodability
!python -m src.analysis.representation_robustness
!python -m src.interpretability.bottleneck_layer
!python -m src.interpretability.bootstrap_direction_stability
!python -m src.interpretability.bootstrap_cross_branch_difference
!python -m src.interpretability.paired_deep_layer_stability_test --seed 20260904
!python -m src.analysis.summarize_probe_findings
!python -m src.analysis.summarize_cross_branch


## 10. Descriptive causal/steering summaries for the new files

In [ ]:
import glob
for stage in ['M3_direct', 'M3_alt', 'M3_direct_alt']:
    f = f'results/raw/causal_ablation_v2_{stage}_L24-28.json'
    get_ipython().system(f'python -m src.analysis.summarize_causal_ablation --file {f}')
    get_ipython().system(f'python -m src.analysis.mcnemar_causal_ablation --file {f} --conditions {stage}_baseline {stage}_ablated_AD')
    get_ipython().system(f'python -m src.analysis.bootstrap_causal_effect --file {f} --quadrant A --category refusal')

for f in glob.glob('results/raw/steering_v2_M3_direct*_QABCD.json'):
    get_ipython().system(f'python -m src.analysis.summarize_steering --file {f}')


## Done
Send back `results/summaries/confirmatory_endpoints.json` and the printed output of steps 9-10 for interpretation. `results/human_review/packet.json` is ready for annotation using `docs/human_review_instructions.md`.

## 11. Build the human-review annotation packet
Needs the judge output from step 7. `--key-out` is written to Drive (outside the git repo) since it must never be committed - it's the blinding key.

## Done
Send back `results/summaries/confirmatory_endpoints.json` and the printed output of cells 9-10 for interpretation.